In [1]:
import pickle

with open('/mnt/nvme0n1/PM/PointMILFigures/3DFigures/Transformer_Positional/cell_0.pkl', 'rb') as f:
    data = pickle.load(f)

In [2]:
data

[{'CellID': 0,
  'Model': 'conjunctive',
  'Fold': 1,
  'Interpretation': array([1.9407982e-07, 1.0000000e+00, 9.9999988e-01, ..., 1.2264421e-03,
         4.2586966e-15, 1.0000000e+00], dtype=float32),
  'GT': array([0, 1, 0, ..., 1, 0, 1]),
  'PointCloud': array([[-0.23143438, -0.01113182, -0.20442003, -0.465868  , -0.694355  ,
          -0.548487  ],
         [ 0.632214  , -0.16393583, -0.22251184,  0.714693  , -0.266304  ,
          -0.646757  ],
         [ 0.09394956, -0.21913633,  0.06409786, -0.359943  , -0.90362   ,
           0.232191  ],
         ...,
         [ 0.29331434,  0.04940864, -0.09455425, -0.015973  ,  0.998863  ,
          -0.044918  ],
         [-0.11352655, -0.04153817, -0.21264929,  0.133581  , -0.580738  ,
          -0.803056  ],
         [ 0.5913684 , -0.3084144 , -0.06968509,  0.690676  , -0.665428  ,
          -0.283146  ]], dtype=float32),
  'Predicted': 1,
  'Truth': array([1])}]

In [4]:
# Convert the discovered structure to the demo JSON format.
import pickle, json, numpy as np

src_path = '/mnt/nvme0n1/PM/PointMILFigures/3DFigures/Transformer_Positional/cell_0.pkl'
out_path = "/home/mvries/Documents/Sentinal4D/GitHub/PointMIL/assets/samples/intra_cell_demo.json"

with open(src_path, "rb") as f:
    data_list = pickle.load(f)

rec = data_list[0]
P = np.asarray(rec["PointCloud"], dtype=np.float64)[:, :3]  # Nx3
interp = np.asarray(rec["Interpretation"], dtype=np.float64).reshape(-1)

# Sanity checks / normalisation
N = P.shape[0]
if interp.size != N:
    raise ValueError(f"Interpretation length {interp.size} != points {N}")

# Unit-sphere normalisation (keeps current scale if already normalised)
c = P.mean(axis=0, keepdims=True)
Q = P - c
r = np.linalg.norm(Q, axis=1).max() or 1.0
Q = Q / r

# Ensure [0,1] range for importance
v = interp
lo, hi = np.nanmin(v), np.nanmax(v)
if hi - lo < 1e-12:
    v01 = np.zeros_like(v)
else:
    v01 = (v - lo) / (hi - lo)

pred_label = str(rec.get("Predicted", "unknown"))

sample = {
    "points": [[float(x), float(y), float(z)] for x, y, z in Q],
    "classes": [f"Predicted={pred_label}"],
    "scores": {f"Predicted={pred_label}": v01.astype(float).tolist()},
    "pred": {"label": pred_label}
}

with open(out_path, "w") as f:
    json.dump(sample, f)

print({
    "output": out_path,
    "num_points": N,
    "class_name": f"Predicted={pred_label}",
    "head_points": sample["points"][:3],
    "head_scores": sample["scores"][f"Predicted={pred_label}"][:5]
})


{'output': '/home/mvries/Documents/Sentinal4D/GitHub/PointMIL/assets/samples/intra_cell_demo.json', 'num_points': 1024, 'class_name': 'Predicted=1', 'head_points': [[-0.23143533138131853, -0.01113895730691534, -0.20442113196141787], [0.6322165480723176, -0.16394358056879, -0.22251301789748926], [0.09394992409267833, -0.21914430620707326, 0.06409784364474817]], 'head_scores': [1.940798171062852e-07, 1.0, 0.9999998807907104, 6.752737249371421e-07, 1.3637709871442381e-14]}
